# Assignment 5: GRPO

<b style="color:red;bold;">Due date: April 1st, 2026</b>

## 0. Preliminary (PLEASE READ THIS CAREFULLY)

### Libraries

The assignment mainly relies on a library developped by Huggingface:

- `transformers`: A comprehensive library that provides state-of-the-art pre-trained models for Natural Language Processing (NLP) and other tasks, offering easy-to-use interfaces for implementing, and fine-tuning a wide range of transformer-based architectures.


In case you are unfamiliar with these libraries, you can consult these resources:

- [Stanford's CS224N transformers tutorial](https://colab.research.google.com/drive/1pxc-ehTtnVM72-NViET_D2ZqOlpOi2LH?usp=sharing)


### (!New!) Required Reading Material

Compared to other assignments, you will need to go over the following reading material:

- Video: [Stanford CME295 Transformers & LLMs | Autumn 2025 | Lecture 6 - LLM Reasoning](https://www.youtube.com/watch?v=k5Fh-UgTuCo&list=PLoROMvodv4rObv1FMizXqumgVVdzX4_05&index=5)

The video provides a comprehensive overview of how reasoning is enabled in LLMs. **The quiz will include questions related to some of the concepts addressed there**.

### Model

In this assignment you will use the Qwen3-0.6B model. It roughly has 600M parameters. Compared to other models of similar size, it performs better, with improved reasoning and instruction-following capabilities. You can find the model on Huggingface's [hub](https://huggingface.co/Qwen/Qwen3-0.6B). You don't need to set `enable_thinking` for tokenization and model loading.


### Execution Environment

It is strongly recommended to use `Google Colab` with the hardware accelerator set to `T4 GPU`, as it enables faster computation and allows models to be loaded without encountering CUDA out-of-memory errors. In case you encountered CUDA out-of-memory error, you can reduce the number of *steps*, *iterations*, and *G*.


### Implementation Guide
- You are tasked with filling in the sections marked as **TODO** and indicated by **ellipses (...)**.
- Please ensure that the output of the final cell is preserved, as it is required to observe the model's behavior and for marking purposes.




---

# GRPO (Group Relative Policy Optimization) — an RL algorithm

DeepSeek used **GRPO** to trigger emergent **reasoning**, such as chain-of-thought, by reinforcing the best solutions from groups of internally generated outputs. This enables the model to learn complex logic purely via **reinforcement learning**, eliminating the need for human-labeled reasoning data.

So, this assignment requires implementing **GRPO** to align language models with a reward function without needing a separate Value network (Critic). You will train a small LLM to solve **grade-school math word problems** using the objective function below.

---

## 1. What is GRPO and Why it Helps

**GRPO** (Group Relative Policy Optimization) is an efficiency-focused variant of PPO. Instead of training a separate "Critic" model to estimate how good a response is (which doubles memory usage), GRPO uses the **group average** as the baseline.

**The Workflow:**

1. **Sample a Group:** For every prompt , generate  different outputs  using the old policy.
2. **Score:** Calculate the reward  for each output.
3. **Relative Advantage:** Determine how much better or worse each output is compared to the **average of that specific group**.
4. **Update:** Optimize the policy to increase the probability of the "good" outputs and decrease the "bad" ones, while keeping the model stable.

**Why it helps:**

* **Memory Efficient:** No need for a Value Model (Critic), saving massive amounts of VRAM.
* **Emergent Reasoning**: By forcing the model to beat its own average, it learns that thinking step-by-step is the only reliable strategy to secure a reward, whereas random guessing usually fails against the group.
* **Self-Normalizing:** By comparing samples against each other, the model learns regardless of whether the absolute rewards are generally high or low.

---
2. The GRPO Objective (The Formula)$$\mathcal{J}_{\text{GRPO}}(\theta)=
\Bigg[
\frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_i|}\sum_{t=1}^{|o_i|}
\Big\{
\min\Big[
\frac{\pi_{\theta}(o_{i,t})}{\pi_{\theta_{\text{old}}}(o_{i,t})} \cdot \hat{A}_{i,t},\,\,
\text{clip}(\frac{\pi_{\theta}(o_{i,t})}{\pi_{\theta_{\text{old}}}(o_{i,t})}, 1-\epsilon, 1+\epsilon) \cdot \hat{A}_{i,t}
\Big]
-\beta D_{\text{KL}}\big[\pi_{\theta}\|\pi_{\text{ref}}\big]
\Big\}
\Bigg]$$
Key Symbols & Terms
- $G$: Group size. The number of outputs generated per prompt.
- $\pi_{\theta}$ / $\pi_{\theta_{old}}$: The trainable policy vs. the frozen version used for data generation.
- $\text{ratio}$: $\frac{\pi_{\theta}(o_{i,t})}{\pi_{\theta_{\text{old}}}(o_{i,t})}$. Probability change since sampling.
- $\epsilon$ (Epsilon): Clipping parameter (0.2). Prevents updates from being too drastic.
- $\beta$ (Beta): Coefficient for the KL penalty (0.04). Controls how much we punish drift.

## 3. How It Works: The Three Models

To implement this, you manage three distinct versions of the model:

1. **Policy Model ($\pi_{\theta}$):**
* **Role:** The "Student." The only model that is actually trained.
* **Code:** `model`. Requires gradients.


2. **Old Policy Model ($\pi_{\theta_{old}}$):**
* **Role:** The "Data Generator." A frozen snapshot of the student. It generates the  samples and calculates the probability denominators.
* **Code:** `old_model`. Synced with `model` after every update step.


3. **Reference Model ($\pi_{ref}$):**
* **Role:** The "Anchor." A completely frozen copy of the initial model. Used to calculate KL divergence to ensure the model doesn't forget its primary behavior while learning Math.
* **Code:** `ref_model`. Never updated.



---

## 4. The "Group Relative" Advantage

Standard RL requires a Critic to guess the baseline. GRPO calculates the baseline mathematically using the group statistics.

For each output $i$ in the group of $G$ outputs:$$\hat{A}_i = \frac{r_i - \text{mean}(r_1...r_G)}{\text{std}(r_1...r_G) + \delta}$$

* **Result:** If a specific output  has a reward higher than the group average,  is positive (encourage). If lower, it is negative (discourage).
* **Broadcasting:** This advantage is calculated once per **completion**, but assigned to **every token** in that completion.

---

## 5. The Dataset & Reward Function

* **Problem:** Mathematical Reasoning using the **GSM8K** dataset.
* **Goal:** The model must reason step-by-step and output the final answer in a specific format.

**The Reward Logic (`calculate_reward`):**
The reward determines exactly what behavior the model learns.

1. **Format Bonus (+0.2):** Awarded if the model successfully uses `<answer>...</answer>` tags. This shapes the structural behavior.
2. **Correctness Bonus (+1.0):** Awarded if the content inside the tags matches the Gold Answer exactly (spaces/commas ignored).


---

## 6. Key Mechanisms: Clipping & KL

### The Clipping Term
$$\text{clip}\left(\text{ratio}, 1-\epsilon, 1+\epsilon\right)$$

This is the "safety harness." If the model updates its weights so much that a token becomes much more likely than it was in the Old Policy, the ratio explodes. The clipping term ignores advantages from such massive shifts, keeping the training stable and within a "Trust Region."

### The KL Divergence (Top-K)
$$-\beta D_{\text{KL}}[\pi_{\theta}\|\pi_{\text{ref}}]$$

This penalty discourages "Reward Hacking" (producing gibberish that technically satisfies the reward function but isn't valid text).




---

# NOTE (PLEASE READ THIS CAREFULLY)

---
- Note that a true GRPO-style approach first requires cold supervised fine-tuning. This step enables the model to comply with the required format and learn the expected generation behavior. However, since fine-tuning is beyond the scope of this assignment, it is skipped.

- Also note that you do not need to understand the KL-divergence functions, as this is a variant of KL-divergence that does not raise out-of-memory errors.

- Additionally, you may not observe a clear loss pattern, which is acceptable. The current model is very small, and its ability to generate correct solutions to mathematical problems is limited. Therefore, your grade will be evaluated solely based on the correctness and logical soundness of your implementation of the **TODO** section.

## INITIAL SETTING

In [1]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import re
import gc

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


model_name = "Qwen/Qwen3-0.6B"
G = 4               # Group size: number of samples per prompt
EPSILON = 0.2       # Clipping parameter for PPO
BETA = 0.04         # Coefficient for the KL penalty
LEARNING_RATE = 2e-5
NUM_STEPS = 20      # Number of training steps (batches)
MAX_NEW_TOKENS = 400
NUM_ITERATIONS = 3

Using device: cuda


## Loading Models

In [2]:
print("Loading models...")
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# TODO: Initialize three models based on the equation: π_θ, π_θ_old, and π_ref.
# For the model(s) that should be updated, call model.train() and
# model.gradient_checkpointing_enable() to reduce CUDA memory usage.
# For the model(s) that are not updated during training, set model.eval()
# and disable gradient computation.
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.train()
model.gradient_checkpointing_enable()

ref_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

old_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
old_model.eval()
for param in old_model.parameters():
    param.requires_grad = False
print("Models loaded (Policy, Reference, Old Policy).")

Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models loaded (Policy, Reference, Old Policy).


## Prompting and Parsing Output for Reward

In [3]:
def format_prompt(question):
    return (
        f"Question: {question}\n"
        f"Think step by step to solve this. Output the final answer inside <answer> tags: <answer>...</answer>.\n"
        f"Answer:"
    )

# TODO: Take a look at https://huggingface.co/datasets/openai/gsm8k
# Load the dataset using the train split
dataset = load_dataset("openai/gsm8k", "main", split="train")

def get_gold_answer(text):
    # TODO: Inspect how the answer is written in each sample and return it as a string
    # HINT: The final answer always appears after "####"
    parts = text.split("####")
    if len(parts) > 1:
        return parts[1].strip()
    return ""

def calculate_reward(generated_text, gold_answer):
    # TODO: Find the last <answer>...</answer> tag.
    # If found, reward the model with +0.2.
    # Also, if the generated answer exactly matches the gold answer, reward the model with +1.
    reward = 0.0

    # Find all <answer>...</answer> tags and take the last one
    matches = re.findall(r"<answer>\s*(.*?)\s*</answer>", generated_text)

    if matches:
        # Format bonus for using the correct tags
        reward += 0.2

        # Extract the last match
        extracted_answer = matches[-1].strip()

        # Clean both answers for comparison (remove spaces and commas)
        clean_extracted = extracted_answer.replace(" ", "").replace(",", "")
        clean_gold = gold_answer.replace(" ", "").replace(",", "")

        # Correctness bonus if they match
        if clean_extracted == clean_gold:
            reward += 1.0

    return reward

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

## FUNCTION ONLY FOR DEBUG PURPOSE

In [4]:
def debug_step(step, prompt_str, generated_texts, gold_ans):
    print("\n" + "=" * 30)
    print(f"DEBUG STEP {step}")
    print(f"Prompt: {prompt_str[:50]}...")  # Print start of prompt
    print(f"Generated (Sample 0): {generated_texts[0]}")  # First generation
    print(f"TRUE SOLUTION is: {gold_ans}")

    # Check if regex matches
    matches = re.findall(r"<answer>\s*(.*?)\s*</answer>", generated_texts[0])
    if matches:
        print(f"MATCH FOUND: {matches[-1]}")
    else:
        print("NO MATCH FOUND - Reward will be 0.0")

    print("=" * 30 + "\n")

## Helper Function to Compute Probabilities of Model Logits

In [5]:
def get_batch_log_probs(logits, labels, pad_token_id):
    """
    Compute per-token log-probabilities of the target labels for a batch,
    while masking out padding tokens.

    Args:
        logits (torch.Tensor):
            Model outputs of shape (batch_size, seq_len, vocab_size).
        labels (torch.Tensor):
            Ground-truth token IDs of shape (batch_size, seq_len).
        pad_token_id (int):
            Token ID used for padding, which should be ignored in the loss.

    Returns:
        target_log_probs (torch.Tensor):
            Log-probabilities of the correct next tokens, with padding
            positions zeroed out. Shape: (batch_size, seq_len - 1).
        mask (torch.Tensor):
            Float mask indicating non-padding positions (1.0 = valid token,
            0.0 = padding). Shape: (batch_size, seq_len - 1).
    """

    # TODO: Remove the last time step so each position predicts the next token
    # Shape: (batch_size, seq_len - 1, vocab_size)
    shift_logits = logits[:, :-1, :]

    # TODO: Remove the first token so labels are aligned with shifted logits
    # Shape: (batch_size, seq_len - 1)
    shift_labels = labels[:, 1:]

    # Create a mask to ignore padding tokens in the loss computation
    # 1.0 for valid tokens, 0.0 for padding
    mask = (shift_labels != pad_token_id).float()

    # Convert logits to log-probabilities over the vocabulary
    log_probs_all = F.log_softmax(shift_logits, dim=-1)

    # TODO: Select the log-probability corresponding to the target label at each position
    # Final shape should be: (batch_size, seq_len - 1)
    # HINT: use torch.gather function
    target_log_probs = log_probs_all.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

    # Zero out log-probabilities at padding positions
    return target_log_probs * mask, mask

## KL-Divergence (DON'T MODIFY)

In [6]:
def kl_topk_tokenwise(logits_theta, logits_ref, k=32):
    """
    Returns tokenwise KL(pi_theta || pi_ref) over the TOP-K tokens of pi_theta.
    logits_*: [B, T, V]
    output:   [B, T]
    """
    # top-k tokens under current policy
    topv, topi = torch.topk(logits_theta, k, dim=-1)             # [B,T,K]
    ref_top = torch.gather(logits_ref, -1, topi)                 # [B,T,K]

    logp_t = F.log_softmax(topv.float(), dim=-1)                 # [B,T,K]
    logp_r = F.log_softmax(ref_top.float(), dim=-1)              # [B,T,K]
    p_t = logp_t.exp()

    kl = (p_t * (logp_t - logp_r)).sum(dim=-1)                   # [B,T]
    return kl



def kl_topk_completion_aligned(
    logits_theta: torch.Tensor,
    logits_ref_full: torch.Tensor,
    log_probs_theta: torch.Tensor,
    prompt_len: int,
    k: int = 32,
) -> torch.Tensor:
    """
    Build tokenwise KL penalty tensor aligned with get_batch_log_probs output shape [B, L-1],
    but only computed on completion tokens (shifted index >= prompt_len-1) using Top-K KL.
    """
    comp_start = prompt_len - 1  # shifted index where completion starts

    # shift logits to match get_batch_log_probs convention
    logits_theta_shift = logits_theta[:, :-1, :]
    logits_ref_shift   = logits_ref_full[:, :-1, :]

    # compute KL only on completion tokens (saves compute)
    kl_comp = kl_topk_tokenwise(
        logits_theta_shift[:, comp_start:, :],
        logits_ref_shift[:, comp_start:, :],
        k=k
    )  # [B, Tcomp]

    # put it back into full [B, L-1] shape
    kl_tok = torch.zeros_like(log_probs_theta)
    kl_tok[:, comp_start:] = kl_comp

    return kl_tok

## Main Loop

In [7]:

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print("Starting GRPO training...")

for step in range(NUM_STEPS):
    # Memory Cleanup
    torch.cuda.empty_cache()
    gc.collect()

    # Select prompt
    item = dataset[step]
    prompt_str = format_prompt(item['question'])
    gold_ans = get_gold_answer(item['answer'])

    # TODO: Replicate prompt_str G times and tokenize it
    prompts = [prompt_str] * G
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(device)
    prompt_len = inputs['input_ids'].shape[1]

    # TODO: Perform sampling (generation) using old_model.
    # Set temperature to 0.7 and do_sample to True.
    with torch.no_grad():
        outputs = old_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Process rewards
    generated_tokens_only = outputs[:, prompt_len:]
    generated_texts = tokenizer.batch_decode(generated_tokens_only, skip_special_tokens=True)

    # TODO: The following function is only for debugging purposes.
    # Uncomment it if you want to observe the model's behavior.

    # debug_step(
    #     step=step,
    #     prompt_str=prompt_str,
    #     generated_texts=generated_texts,
    #     gold_ans=gold_ans
    # )



    rewards = []
    for i in range(G):
        r = calculate_reward(generated_texts[i], gold_ans)
        rewards.append(r)

    rewards_tensor = torch.tensor(rewards, dtype=torch.float32).to(device)

    # TODO: Compute the Group Relative Advantage based on the formula
    # provided in the instruction markdown cell: r - mean(rewards) / std(rewards) + 1e-4
    # The final shape should be [G, 1].
    mean_reward = rewards_tensor.mean()
    std_reward = rewards_tensor.std()
    advantages = ((rewards_tensor - mean_reward) / (std_reward + 1e-4)).unsqueeze(1)

    print(f"Step {step+1}: Mean Reward={rewards_tensor.mean().item():.2f}")


    with torch.no_grad():
        # TODO: Fill in the ...
        # Old Policy Log Probs (Denominator of Ratio)
        outputs_old = old_model(outputs, attention_mask=(outputs != tokenizer.pad_token_id))
        log_probs_old, full_mask = get_batch_log_probs(outputs_old.logits, outputs, tokenizer.pad_token_id)

        # TODO: Fill in the ...
        # Reference Model Log Probs (for KL)
        outputs_ref = ref_model(outputs, attention_mask=(outputs != tokenizer.pad_token_id))

    # Create Completion Mask
    completion_mask = torch.zeros_like(full_mask)
    # TODO: Mask out prompt tokens so that only completion tokens contribute to the loss
    # HINT: completion_mask[FILL HERE] = 1
    completion_mask[:, prompt_len-1:] = 1
    final_mask = full_mask * completion_mask


    model.train()

    # Update the policy multiple times on the same batch
    for i in range(NUM_ITERATIONS):
        # TODO: Fill in the ...
        # Forward pass with current policy (pi_theta)
        outputs_theta = model(outputs, attention_mask=(outputs != tokenizer.pad_token_id))
        logits_theta = outputs_theta.logits
        log_probs_theta, _ = get_batch_log_probs(logits_theta, outputs, tokenizer.pad_token_id)

        # Ratio: pi_theta / pi_old
        # Since probabilities are in log space:
        # exp(log(A) - log(B)) = A / B
        ratios = torch.exp(log_probs_theta - log_probs_old)

        # TODO: Compute the minimum term in the GRPO equation:
        # min(
        #   (pi_theta / pi_old) * A_i,t,
        #   clip(pi_theta / pi_old, 1 - ε, 1 + ε) * A_i,t
        # )
        clipped_ratios = torch.clamp(ratios, 1 - EPSILON, 1 + EPSILON)
        ppo_obj = torch.min(ratios * advantages, clipped_ratios * advantages)

        # KL Penalty
        kl_tok = kl_topk_completion_aligned(
            logits_theta=logits_theta,          # [B, L, V]
            logits_ref_full=outputs_ref.logits,    # [B, L, V]
            log_probs_theta=log_probs_theta,    # [B, L-1]
            prompt_len=prompt_len,              # int
            k=32
        )
        kl_penalty = BETA * kl_tok

        # Total Objective
        objective = ppo_obj - kl_penalty
        objective = objective * final_mask

        # TODO: Masking and averaging
        # Perform the terms before the braces in the equation:
        # 1/G Σ 1/|o_i| Σ :
        # 1. Compute the number of newly generated tokens (HINT: use final_mask)
        # 2. Sum the objective and divide by the token count
        # 3. Average across the group dimension (HINT: use mean)
        # 4. Since PyTorch minimizes losses, negate the objective to convert
        #    maximization into minimization. (The negative sign is already there, keep it as it is)
        token_counts = final_mask.sum(dim=1)
        seq_objective = objective.sum(dim=1) / token_counts
        loss = -seq_objective.mean()

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"  Loss: {loss.item():.4f}")

    # TODO: Sync Old Policy
    # Update old_model to match the current policy for the next step
    # HINT: Use load_state_dict and state_dict
    old_model.load_state_dict(model.state_dict())

print("Training finished.")

Starting GRPO training...
Step 1: Mean Reward=0.70


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  Loss: 0.0000
  Loss: -0.0197
  Loss: -0.0412
Step 2: Mean Reward=0.20
  Loss: 0.0018
  Loss: 0.0014
  Loss: 0.0012
Step 3: Mean Reward=0.70
  Loss: 0.0022
  Loss: -0.0207
  Loss: -0.0326
Step 4: Mean Reward=0.15
  Loss: 0.0032
  Loss: -0.0147
  Loss: -0.0263
Step 5: Mean Reward=0.20
  Loss: 0.0035
  Loss: 0.0036
  Loss: 0.0036
Step 6: Mean Reward=0.20
  Loss: 0.0029
  Loss: 0.0029
  Loss: 0.0028
Step 7: Mean Reward=0.70
  Loss: 0.0047
  Loss: -0.0146
  Loss: -0.0307
Step 8: Mean Reward=0.70
  Loss: 0.0041
  Loss: -0.0190
  Loss: -0.0351
Step 9: Mean Reward=0.20
  Loss: 0.0041
  Loss: 0.0041
  Loss: 0.0040
Step 10: Mean Reward=0.20
  Loss: 0.0040
  Loss: 0.0039
  Loss: 0.0038
Step 11: Mean Reward=0.20
  Loss: 0.0027
  Loss: 0.0026
  Loss: 0.0025
Step 12: Mean Reward=0.20
  Loss: 0.0032
  Loss: 0.0031
  Loss: 0.0030
Step 13: Mean Reward=0.45
  Loss: 0.0063
  Loss: -0.0091
  Loss: -0.0205
Step 14: Mean Reward=0.20
  Loss: 0.0044
  Loss: 0.0044
  Loss: 0.0044
Step 15: Mean Reward=0.45
  

# Rubric:

#### Loading models: /3

#### Loading Dataset, extractiong the answer, and completing the reward function: /5

#### `get_batch_log_probs`: /4

#### Last Cell (Main Loop):
- TODO - Replicate prompt_str G times and tokenize it: /1
- TODO - Perform sampling (generation) using old_model: /1
- TODO - Compute relative advantage: /2
- TODO - Old policy log probs and reference policy log probs: /1
- TODO - Completion mask: /1
- TODO - Forward pass with current policy: /1
- TODO - ppo_obj: /2
- TODO - Masking and averaging: /2
- TODO - Sync Old Policy: /1

### Overall: /24
